In [1]:
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback
import torch
from datasets import Dataset
import pandas as pd
import gc
import json

In [2]:
dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'
cv_path= "Model_dataset/cv.json"
with open(cv_path, "r") as file:
    cv_data= json.load(file)


#qa model
qa_type_model_name= 't5-base'
qa_type_model_result= '.temp/model_results/fine_tuned_question_answer_model-base'
qa_type_model= '.temp/model/fine_tuned_question_answer_model-base'



In [3]:
cv_data.keys()

dict_keys(['current_ctc', 'expected_ctc', 'personal_information', 'education', 'working_experince', 'skills', 'availability', 'others'])

### Preprocessing

In [4]:
df= pd.read_csv(dataset_path)
df= df[["question", "question_type", "answer"]]
df["answer"]= df["answer"].fillna("")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
 2   answer         1644 non-null   object
dtypes: object(3)
memory usage: 38.7+ KB


In [5]:
df["context"] = df["question_type"].map(cv_data)

In [6]:
df= df.sample(frac=1).reset_index(drop=True)
df.head()

,question,question_type,answer,context
0,How do you handle logging and monitoring in yo...,working_experince,"Logging with Airflow, monitoring with AWS Clou...",Total Work Experience: 2+ years (including int...
1,Would you prefer a hybrid model over full WFO?,availability,Yes,Availability for Interviews\nI am available fo...
2,Have you ever built a mentorship program from ...,others,Stay flexible with team direction.,I thrive in dynamic and collaborative work env...
3,How do you prefer to be addressed?,personal_information,Manab,Full Name: Manab Boro\nEmail Address: mboro497...
4,How flexible are you with changes in work prio...,availability,Flexible,Availability for Interviews\nI am available fo...


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
 2   answer         1644 non-null   object
 3   context        1644 non-null   object
dtypes: object(4)
memory usage: 51.5+ KB


### Retraing Preparations:

In [8]:
# Preprocess data
def preprocess_data(row):
    input_text = f"question: {row['question']} context: {row['context']}"
    target_text = row['answer']
    return {"input_text": input_text, "target_text": target_text}

processed_data = df.apply(preprocess_data, axis=1)
dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

In [9]:
# Split data into train and test
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

In [10]:
tokenizer = T5Tokenizer.from_pretrained(qa_type_model_name)

def tokenize_data(example):
    input_encodings = tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=512)
    target_encodings = tokenizer(example["target_text"], truncation=True, padding="max_length", max_length=128)
    input_encodings["labels"] = target_encodings["input_ids"]
    return input_encodings

train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/1479 [00:00<?, ? examples/s]

Map:   0%|          | 0/165 [00:00<?, ? examples/s]

# Train

In [11]:
model = T5ForConditionalGeneration.from_pretrained(qa_type_model_name)

In [12]:
# Training arguments
training_args = TrainingArguments(
    output_dir=qa_type_model_result,  # Output directory
    eval_strategy="epoch",        # Evaluate every epoch
    save_strategy="epoch",              # Save every epoch
    learning_rate=3e-5,                 # Learning rate
    num_train_epochs=50,                # Number of training epochs
    per_device_train_batch_size=2,     # Batch size during training
    per_device_eval_batch_size=2,      # Batch size during evaluation
    gradient_accumulation_steps=1,      # Gradient accumulation steps
    logging_dir="./logs",               # Directory for logs
    logging_steps=10,                   # Log every 10 steps
    load_best_model_at_end=True,        # Load best model at the end of training
    save_total_limit=4,                 # Limit the number of saved checkpoints
    # fp16=True,
    use_cpu= True
)


In [13]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Clearing memory before training
torch.cuda.empty_cache()
gc.collect()

# Train the model
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.060400,0.049494
2,0.077000,0.046668
3,0.024600,0.044513
4,0.024900,0.042666
5,0.036100,0.043409
6,0.022500,0.045185


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4440, training_loss=0.11798030333006167, metrics={'train_runtime': 47044.8061, 'train_samples_per_second': 1.572, 'train_steps_per_second': 0.786, 'total_flos': 5403892320829440.0, 'train_loss': 0.11798030333006167, 'epoch': 6.0})

In [14]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.042665526270866394,
 'eval_runtime': 189.5675,
 'eval_samples_per_second': 0.87,
 'eval_steps_per_second': 0.438,
 'epoch': 6.0}

In [15]:
trainer.save_model(qa_type_model)
tokenizer.save_pretrained(qa_type_model)

('.temp/model/fine_tuned_question_answer_model-base/tokenizer_config.json',
 '.temp/model/fine_tuned_question_answer_model-base/special_tokens_map.json',
 '.temp/model/fine_tuned_question_answer_model-base/spiece.model',
 '.temp/model/fine_tuned_question_answer_model-base/added_tokens.json')